# Combined Classifications Strategies analysis

In [1]:
import pandas as pd
from typing import Dict, Any, Union
import re
import numpy as np
import json
from sklearn.linear_model import LinearRegression

## Loading GNN classifications

In [2]:
def enrich_parser_evaluation_metadata(df: pd.DataFrame, drop_filename: bool = False) -> pd.DataFrame:
    """
    Enrich metadata for filenames in parser evaluation CSV.
    """

    program_map = {
        "picohttpparser":          "PicoHTTPParser",
        "cjson":                   "cJSON",
        "http_parser":             "Benoitc_HTTP",
        "parse_xml":               "CParserXML",
        "example":                 "CSimpleJSONParser",
        "network_analyzer":        "Network_Packet_Analyzer",
        "elf-parser":              "ELF_Parser",
        "pcap_parser":             "PCAP_Parser",
        "packcc":                  "Packcc",
        "calc":                    "YACC_Calculator",
    }

    # ---------------------------------------------------
    # Extract correct program using full prefix matching
    # ---------------------------------------------------
    def extract_program(fname: str):
        for prefix, canonical in program_map.items():
            if fname.startswith(prefix):   # <--------- FIXED
                return canonical
        return "UNKNOWN"

    # ---------------------------------------------------
    # Extract optimization level
    # ---------------------------------------------------
    def extract_opt_level(fname: str):
        if "_Ofast" in fname:
            return "Ofast"
        if "_O3" in fname:
            return "O3"
        if "_O2" in fname:
            return "O2"
        if "_O1" in fname:
            return "O1"
        if "_O0" in fname:
            return "O0"
        if "_Os" in fname:
            return "Os"
        return "default"

    # ---------------------------------------------------
    # Extract compiler
    # ---------------------------------------------------
    def extract_compiler(fname: str):
        return "clang" if "_clang" in fname else "gcc"

    df = df.copy()
    df["program"] = df["filename"].apply(extract_program)
    df["opt_level"] = df["filename"].apply(extract_opt_level)
    df["compiler"] = df["filename"].apply(extract_compiler)

    if drop_filename:
        df = df.drop(columns=["filename"])

    return df

In [3]:
gnn_df = pd.read_csv("../Results/Safetorch/GraphSAGE/analyzed_validation_results.csv")
# Shape of the dataframe
print(f"gnn_df.shape: {gnn_df.shape}")
# Display first few rows
gnn_df.head()

gnn_df.shape: (505472, 16)


,filename,name,address,node_shape,edge_shape,true_label,prediction,parser_probability,dataset_len,hidden_dim,dropout,learning_rate,epochs,batch_size,weight_decay,num_heads
0,picohttpparser_3_32.pt,sym.test_chunked_consume_trailer,0x804a260,"(11, 100)","(2, 16)",0,1,0.947381,760,16,0.0,0.1,10,32,0.0,1
1,picohttpparser_3_32.pt,sym.test_chunked_at_once,0x804aa20,"(32, 100)","(2, 44)",0,1,0.576798,760,16,0.0,0.1,10,32,0.0,1
2,picohttpparser_3_32.pt,sym.test_chunked_leftdata,0x804a320,"(15, 100)","(2, 20)",0,0,0.488713,760,16,0.0,0.1,10,32,0.0,1
3,picohttpparser_3_32.pt,sym.note,0x804a020,"(11, 100)","(2, 13)",0,0,0.429781,760,16,0.0,0.1,10,32,0.0,1
4,picohttpparser_3_32.pt,sym.phr_decode_chunked,0x8049d00,"(6, 100)","(2, 5)",1,0,0.022807,760,16,0.0,0.1,10,32,0.0,1


In [4]:
# Read the best parameters from json file
with open("../Results/Safetorch/GraphSAGE/best_params.json", "r") as f:
    best_params = json.load(f)

print(f"Best parameters: {best_params}")

Best parameters: {'batch_size': 64, 'dropout': 0, 'epochs': 10, 'hidden_dim': 16, 'input_dim': 100, 'learning_rate': 0.01, 'output_dim': 2, 'weight_decay': 0.01}


In [5]:
# Now we filter for the best parameters (we take the best model only)
gnn_df_filtered = gnn_df[
    (gnn_df["batch_size"] == best_params["batch_size"]) &
    (gnn_df["dropout"] == best_params["dropout"]) &
    (gnn_df["epochs"] == best_params["epochs"]) &
    (gnn_df["hidden_dim"] == best_params["hidden_dim"]) &
    (gnn_df["learning_rate"] == best_params["learning_rate"]) &
    (gnn_df["weight_decay"] == best_params["weight_decay"])
]


# Shape of the dataframe
print(f"gnn_df_filtered.shape: {gnn_df_filtered.shape}")
# Display first few rows
gnn_df_filtered.head()

gnn_df_filtered.shape: (7898, 16)


,filename,name,address,node_shape,edge_shape,true_label,prediction,parser_probability,dataset_len,hidden_dim,dropout,learning_rate,epochs,batch_size,weight_decay,num_heads
274716,picohttpparser_3_32.pt,sym.test_chunked_consume_trailer,0x804a260,"(11, 100)","(2, 16)",0,1,0.584732,760,16,0.0,0.01,10,64,0.01,1
274717,picohttpparser_3_32.pt,sym.test_chunked_at_once,0x804aa20,"(32, 100)","(2, 44)",0,1,0.640205,760,16,0.0,0.01,10,64,0.01,1
274718,picohttpparser_3_32.pt,sym.test_chunked_leftdata,0x804a320,"(15, 100)","(2, 20)",0,0,0.481506,760,16,0.0,0.01,10,64,0.01,1
274719,picohttpparser_3_32.pt,sym.note,0x804a020,"(11, 100)","(2, 13)",0,0,0.496029,760,16,0.0,0.01,10,64,0.01,1
274720,picohttpparser_3_32.pt,sym.phr_decode_chunked,0x8049d00,"(6, 100)","(2, 5)",1,0,0.117576,760,16,0.0,0.01,10,64,0.01,1


In [6]:
# Drop unnecessary columns: "node_shape", "edge_shape", "dataset_len", "hidden_dim", "dropout", "learning_rate", "epochs", "batch_size", "weight_decay", "num_heads"

gnn_df_filtered = gnn_df_filtered.drop(columns=[
    "node_shape", "edge_shape", "dataset_len", "hidden_dim", "dropout",
    "learning_rate", "epochs", "batch_size", "weight_decay", "num_heads"
])

# Shape of the dataframe
print(f"gnn_df_filtered.shape: {gnn_df_filtered.shape}")
# Display first few rows
gnn_df_filtered.head()

gnn_df_filtered.shape: (7898, 6)


,filename,name,address,true_label,prediction,parser_probability
274716,picohttpparser_3_32.pt,sym.test_chunked_consume_trailer,0x804a260,0,1,0.584732
274717,picohttpparser_3_32.pt,sym.test_chunked_at_once,0x804aa20,0,1,0.640205
274718,picohttpparser_3_32.pt,sym.test_chunked_leftdata,0x804a320,0,0,0.481506
274719,picohttpparser_3_32.pt,sym.note,0x804a020,0,0,0.496029
274720,picohttpparser_3_32.pt,sym.phr_decode_chunked,0x8049d00,1,0,0.117576


In [7]:
final_gnn_df = enrich_parser_evaluation_metadata(df=gnn_df_filtered, drop_filename=True)

# Shape of the dataframe
print(f"final_gnn_df.shape: {final_gnn_df.shape}")
# Display first few rows
final_gnn_df.head()

final_gnn_df.shape: (7898, 8)


,name,address,true_label,prediction,parser_probability,program,opt_level,compiler
274716,sym.test_chunked_consume_trailer,0x804a260,0,1,0.584732,PicoHTTPParser,default,gcc
274717,sym.test_chunked_at_once,0x804aa20,0,1,0.640205,PicoHTTPParser,default,gcc
274718,sym.test_chunked_leftdata,0x804a320,0,0,0.481506,PicoHTTPParser,default,gcc
274719,sym.note,0x804a020,0,0,0.496029,PicoHTTPParser,default,gcc
274720,sym.phr_decode_chunked,0x8049d00,1,0,0.117576,PicoHTTPParser,default,gcc


In [8]:
# List of unique programs
programs = final_gnn_df["program"].unique()
programs

array(['PicoHTTPParser', 'Benoitc_HTTP', 'CSimpleJSONParser', 'cJSON',
       'CParserXML', 'YACC_Calculator', 'Packcc', 'ELF_Parser',
       'Network_Packet_Analyzer', 'PCAP_Parser'], dtype=object)

## Loading LLM classifications

In [9]:
def enrich_program_metadata(df: pd.DataFrame, drop_filename: bool = False) -> pd.DataFrame:
    """
    Parse the 'filename' field and add:
        - program: canonical program family (10 values)
        - opt_level: O0/O1/O2/O3/Os/Ofast/default
        - compiler: gcc or clang

    Parameters
    ----------
    df : DataFrame
        Input dataframe containing column 'filename'.
    drop_filename : bool
        If True, removes the original 'filename' column.

    Returns
    -------
    DataFrame
        Enriched DataFrame with new columns.
    """

    # --------------------------------------------
    # 1) Canonical Program Family Mapping
    # --------------------------------------------
    program_map = {
        "picohttpparser":          "PicoHTTPParser",
        "cjson":                   "cJSON",
        "http_parser":             "Benoitc_HTTP",
        "parse_xml":               "CParserXML",
        "example":                 "CSimpleJSONParser",
        "network_analyzer":        "Network_Packet_Analyzer",
        "elf-parser":              "ELF_Parser",
        "pcap_parser":             "PCAP_Parser",
        "packcc":                  "Packcc",
        "calc":                    "YACC_Calculator",   # calc is also from YACC grammar
    }

    # --------------------------------------------
    # 2) Function to extract canonical program name
    # --------------------------------------------
    def extract_program(fname):
        for prefix, canonical in program_map.items():
            if fname.startswith(prefix):
                return canonical
        return "UNKNOWN"

    # --------------------------------------------
    # 3) Extract optimization level
    # --------------------------------------------
    def extract_opt_level(fname):
        # Order matters: check Ofast before O3, etc
        if "_Ofast" in fname:
            return "Ofast"
        if "_O3" in fname:
            return "O3"
        if "_O2" in fname:
            return "O2"
        if "_O1" in fname:
            return "O1"
        if "_O0" in fname:
            return "O0"
        if "_Os" in fname:
            return "Os"
        return "default"

    # --------------------------------------------
    # 4) Extract compiler
    # --------------------------------------------
    def extract_compiler(fname):
        return "clang" if "_clang" in fname else "gcc"

    # ------------------------------------------------
    # Apply all parsing functions
    # ------------------------------------------------
    df = df.copy()
    df["program"] = df["filename"].apply(extract_program)
    df["opt_level"] = df["filename"].apply(extract_opt_level)
    df["compiler"] = df["filename"].apply(extract_compiler)

    # Remove original filename if requested
    if drop_filename:
        df = df.drop(columns=["filename"])

    return df

In [10]:
def compute_program_recall(df: pd.DataFrame, program_name: str, code_column: str) -> Dict[str, Any]:
    """
    Compute recall and additional statistics for a specific *program* (not filename).

    Parameters
    ----------
    df : pandas.DataFrame
        The entire prediction dataset loaded from the CSV.
    program_name : str
        Canonical name of the program (matches column 'program').

    Returns
    -------
    Dict[str, Any]
        Dictionary with recall and detailed statistics.
    """

    # Filter rows belonging to this program
    prog_df = df[df["program"] == program_name]

    if prog_df.empty:
        raise ValueError(f"No entries found for program '{program_name}'")

    # ----------------------------------------------------------
    # Convert predicted & true labels to numeric, coercing errors
    # non-numeric labels become NaN and will be excluded automatically
    # ----------------------------------------------------------
    true = pd.to_numeric(prog_df["true_label"], errors="coerce")
    pred = pd.to_numeric(prog_df["predicted_label"], errors="coerce")

    # ----------------------------------------------------------
    # Missing code ("" or NaN)
    # ----------------------------------------------------------
    missing_code = ((prog_df[code_column].isna()) |
                      (prog_df[code_column] == "")).sum()

    # ----------------------------------------------------------
    # Recall computation only on valid numeric entries
    # ----------------------------------------------------------
    valid_mask = (~true.isna()) & (~pred.isna())
    true_valid = true[valid_mask]
    pred_valid = pred[valid_mask]

    tp = ((true_valid == 1) & (pred_valid == 1)).sum()
    actual_pos = (true_valid == 1).sum()

    if actual_pos == 0:
        recall = 1.0
    else:
        recall = tp / actual_pos

    # ----------------------------------------------------------
    # Predicted label statistics
    # ----------------------------------------------------------
    num_pred_0 = (pred_valid == 0).sum()
    num_pred_1 = (pred_valid == 1).sum()
    num_pred_unknown = pred.isna().sum()

    # ----------------------------------------------------------
    # Return statistics dictionary
    # ----------------------------------------------------------
    stats = {
        "program_name": program_name,
        "num_entries": len(prog_df),
        "num_valid_predictions": len(pred_valid),
        "num_missing_code": missing_code,

        # Prediction distribution
        "num_pred_0": num_pred_0,
        "num_pred_1": num_pred_1,
        "num_pred_unknown": num_pred_unknown,

        # Recall metrics
        "true_positives": tp,
        "actual_positives": actual_pos,
        "recall": recall,
    }

    return stats

## Decompiled Code LLM Classifications analysis

In [51]:
llm_df = pd.read_csv("../Results/llm_classification_mistral/global_predictions_decompiled_code.csv")
# Shape of the dataframe
print(f"llm_df.shape: {llm_df.shape}")
# Display first few rows
llm_df.head()

llm_df.shape: (7898, 6)


,filename,function_name,address,decompiled_code,predicted_label,true_label
0,parse_xml,sym.deregister_tm_clones,0x401110,long long deregister_tm_clones()\n{\n if (t...,0,0
1,parse_xml,sym.register_tm_clones,0x401140,long long register_tm_clones()\n{\n if (tru...,0,0
2,parse_xml,sym.__do_global_dtors_aux,0x401180,extern char __bss_start;\n\nlong long __do_glo...,0,0
3,parse_xml,sym.frame_dummy,0x4011b0,long long frame_dummy()\n{\n return registe...,0,0
4,parse_xml,sym.skip_whitespace,0x4011cb,long long skip_whitespace(unsigned long a0)\n{...,0,1


In [52]:
# Remove the filename column and add: program, compiler, opt_level
llm_df = enrich_program_metadata(llm_df, drop_filename=True)

# Shape of the dataframe
print(f"llm_df.shape: {llm_df.shape}")
# Display first few rows
llm_df.head()

llm_df.shape: (7898, 8)


,function_name,address,decompiled_code,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,long long deregister_tm_clones()\n{\n if (t...,0,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,long long register_tm_clones()\n{\n if (tru...,0,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,extern char __bss_start;\n\nlong long __do_glo...,0,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,long long frame_dummy()\n{\n return registe...,0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,long long skip_whitespace(unsigned long a0)\n{...,0,1,CParserXML,default,gcc


In [53]:
# Remove the column "decompiled_code"
llm_df = llm_df.drop(columns=["decompiled_code"])

# Shape of the dataframe
print(f"llm_df.shape: {llm_df.shape}")
# Display first few rows
llm_df.head()

llm_df.shape: (7898, 7)


,function_name,address,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,0,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,0,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,0,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,0,1,CParserXML,default,gcc


In [54]:
# List of unique programs
programs = llm_df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

# Merging both LLM classifications and GNN classifications

In [55]:
# GNN dataframe 

# Shape of the dataframe
print(f"final_gnn_df.shape: {final_gnn_df.shape}")
# Display first few rows
final_gnn_df.head()

final_gnn_df.shape: (7898, 8)


,name,address,true_label,prediction,parser_probability,program,opt_level,compiler
274716,sym.test_chunked_consume_trailer,0x804a260,0,1,0.584732,PicoHTTPParser,default,gcc
274717,sym.test_chunked_at_once,0x804aa20,0,1,0.640205,PicoHTTPParser,default,gcc
274718,sym.test_chunked_leftdata,0x804a320,0,0,0.481506,PicoHTTPParser,default,gcc
274719,sym.note,0x804a020,0,0,0.496029,PicoHTTPParser,default,gcc
274720,sym.phr_decode_chunked,0x8049d00,1,0,0.117576,PicoHTTPParser,default,gcc


In [56]:
# LLM dataframe

# Shape of the dataframe
print(f"llm_df.shape: {llm_df.shape}")
# Display first few rows
llm_df.head()

llm_df.shape: (7898, 7)


,function_name,address,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,0,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,0,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,0,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,0,1,CParserXML,default,gcc


In [57]:
# List of unique programs
programs = llm_df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

In [58]:
# List of unique programs
programs = final_gnn_df["program"].unique()
programs

array(['PicoHTTPParser', 'Benoitc_HTTP', 'CSimpleJSONParser', 'cJSON',
       'CParserXML', 'YACC_Calculator', 'Packcc', 'ELF_Parser',
       'Network_Packet_Analyzer', 'PCAP_Parser'], dtype=object)

In [60]:
def merge_llm_gnn(gnn_df, llm_df):
    # Add a duplicate counter per key to each DF
    gnn_df = gnn_df.copy()
    llm_df = llm_df.copy()

    merge_keys_gnn = ["name", "address", "program", "opt_level", "compiler", "true_label"]
    merge_keys_llm = ["function_name", "address", "program", "opt_level", "compiler", "true_label"]

    gnn_df["dup_idx"] = gnn_df.groupby(merge_keys_gnn).cumcount()
    llm_df["dup_idx"] = llm_df.groupby(merge_keys_llm).cumcount()

    # Now merge 1-to-1 on keys + dup_idx
    merged = pd.merge(
        gnn_df,
        llm_df,
        left_on=merge_keys_gnn + ["dup_idx"],
        right_on=merge_keys_llm + ["dup_idx"],
        how="inner"
    )

    # Cleanup
    merged = merged.drop(columns=["dup_idx", "function_name"])
    merged = merged.rename(columns={
        "prediction": "gnn_predicted_label",
        "predicted_label": "llm_predicted_label"
    })

    return merged

In [61]:
def add_fusion_and_cascade(df: pd.DataFrame,
                            gnn_prob_col: str = "parser_probability",
                            llm_label_col: str = "llm_predicted_label",
                            weighted_fusion_col: str = "weighted_fusion",
                            hybrid_cascade_col: str = "hybrid_cascade",
                            gnn_weight: float = 0.6) -> pd.DataFrame:
    """
    Add two columns to the DataFrame:
    1. Weighted score fusion between GNN probability and LLM prediction
    2. Hybrid cascaded model: GNN first, LLM only when unsure
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing at least the columns:
        - gnn_prob_col (float between 0 and 1)
        - llm_label_col (str or int, LLM prediction, can be '0', '1', or 'unknown')
    gnn_prob_col : str
        Column name for the GNN predicted probability.
    llm_label_col : str
        Column name for the LLM predicted label.
    weighted_fusion_col : str
        Column name to store the weighted fusion score.
    hybrid_cascade_col : str
        Column name to store the hybrid cascaded prediction.
    gnn_weight : float
        Weight for GNN probability in the fusion (0-1). LLM weight = 1 - gnn_weight.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with new columns added.
    """

    def compute_weighted_fusion(row) -> float:
        # Convert LLM label to probability
        llm_label = row[llm_label_col]
        if llm_label in [0, "0"]:
            llm_prob = 0.0
        elif llm_label in [1, "1"]:
            llm_prob = 1.0
        else:  # unknown
            llm_prob = 0.5  # fallback to uniform probability

        gnn_prob = row[gnn_prob_col]
        return gnn_weight * gnn_prob + (1 - gnn_weight) * llm_prob

    def compute_hybrid_cascade(row) -> int:
        gnn_prob = row[gnn_prob_col]
        llm_label = row[llm_label_col]

        # Define "unsure" threshold range for GNN
        unsure_lower = 0.4
        unsure_upper = 0.6

        if unsure_lower <= gnn_prob <= unsure_upper:
            # If LLM prediction is unknown, fallback to GNN
            if llm_label in ["unknown", None]:
                return int(gnn_prob >= 0.5)
            else:
                return int(llm_label)
        else:
            return int(gnn_prob >= 0.5)

    # Add weighted fusion column (probability 0-1)
    df[weighted_fusion_col] = df.apply(compute_weighted_fusion, axis=1)

    # Add hybrid cascaded prediction (hard label 0/1)
    df[hybrid_cascade_col] = df.apply(compute_hybrid_cascade, axis=1)

    return df

In [62]:
def weighted_fusion_lopo(df: pd.DataFrame, verbose: bool = False) -> pd.DataFrame:
    """
    Compute a LOPO (leave-one-program-out) weighted fusion of GNN and LLM predictions
    using linear regression to fit optimal weights on the training programs.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with columns ['program', 'true_label', 'parser_probability', 'llm_predicted_label']
    verbose : bool
        If True, prints progress for each program.

    Returns
    -------
    pd.DataFrame
        DataFrame with new column 'weighted_fusion_opt' containing fused scores [0,1].
    """

    df = df.copy()
    df['weighted_fusion_opt'] = 0.0

    programs = df['program'].unique()

    for program in programs:
        train_df = df[df['program'] != program]
        test_df_idx = df[df['program'] == program].index

        # Prepare training features
        X_train = train_df[['parser_probability', 'llm_predicted_label']].copy()
        # Replace 'unknown' with 0 for LLM labels
        X_train['llm_predicted_label'] = X_train['llm_predicted_label'].replace({'unknown': 0}).astype(float)
        y_train = train_df['true_label']

        # Fit linear regression
        lr = LinearRegression()
        lr.fit(X_train, y_train)

        # Prepare test features
        X_test = df.loc[test_df_idx, ['parser_probability', 'llm_predicted_label']].copy()
        X_test['llm_predicted_label'] = X_test['llm_predicted_label'].replace({'unknown': 0}).astype(float)

        # Compute weighted fusion
        fused_scores = (X_test['parser_probability'] * lr.coef_[0] +
                        X_test['llm_predicted_label'] * lr.coef_[1] +
                        lr.intercept_)
        # Clip to [0,1]
        df.loc[test_df_idx, 'weighted_fusion_opt'] = fused_scores.clip(0, 1)

        if verbose:
            print(f"Program '{program}' processed. Weights: {lr.coef_}, Intercept: {lr.intercept_}")

    return df

In [63]:
# Merge LLM and GNN classifications
df_merged = merge_llm_gnn(llm_df=llm_df, gnn_df=final_gnn_df)

# Shape of the dataframe
print(f"df_merged.shape: {df_merged.shape}")
# Display first few rows
df_merged.head()

df_merged.shape: (7898, 9)


,name,address,true_label,gnn_predicted_label,parser_probability,program,opt_level,compiler,llm_predicted_label
0,sym.test_chunked_consume_trailer,0x804a260,0,1,0.584732,PicoHTTPParser,default,gcc,1
1,sym.test_chunked_at_once,0x804aa20,0,1,0.640205,PicoHTTPParser,default,gcc,1
2,sym.test_chunked_leftdata,0x804a320,0,0,0.481506,PicoHTTPParser,default,gcc,1
3,sym.note,0x804a020,0,0,0.496029,PicoHTTPParser,default,gcc,0
4,sym.phr_decode_chunked,0x8049d00,1,0,0.117576,PicoHTTPParser,default,gcc,0


In [64]:
df_merged = add_fusion_and_cascade(df=df_merged,
                                   gnn_prob_col="parser_probability",
                                   llm_label_col="llm_predicted_label",
                                   weighted_fusion_col="weighted_fusion",
                                   hybrid_cascade_col="hybrid_cascade",
                                   gnn_weight=0.6)

# Shape of the dataframe
print(f"df_merged.shape: {df_merged.shape}")
# Display first few rows
df_merged.head()

df_merged.shape: (7898, 11)


,name,address,true_label,gnn_predicted_label,parser_probability,program,opt_level,compiler,llm_predicted_label,weighted_fusion,hybrid_cascade
0,sym.test_chunked_consume_trailer,0x804a260,0,1,0.584732,PicoHTTPParser,default,gcc,1,0.750839,1
1,sym.test_chunked_at_once,0x804aa20,0,1,0.640205,PicoHTTPParser,default,gcc,1,0.784123,1
2,sym.test_chunked_leftdata,0x804a320,0,0,0.481506,PicoHTTPParser,default,gcc,1,0.688904,1
3,sym.note,0x804a020,0,0,0.496029,PicoHTTPParser,default,gcc,0,0.297617,0
4,sym.phr_decode_chunked,0x8049d00,1,0,0.117576,PicoHTTPParser,default,gcc,0,0.070546,0


In [65]:
# Add a new column "final_prediction" based on weighted fusion thresholded at 0.5
df_merged["final_weighted_fusion_prediction"] = (df_merged["weighted_fusion"] >= 0.5).astype(int)

# Shape of the dataframe
print(f"df_merged.shape: {df_merged.shape}")
# Display first few rows
df_merged.head()

df_merged.shape: (7898, 12)


,name,address,true_label,gnn_predicted_label,parser_probability,program,opt_level,compiler,llm_predicted_label,weighted_fusion,hybrid_cascade,final_weighted_fusion_prediction
0,sym.test_chunked_consume_trailer,0x804a260,0,1,0.584732,PicoHTTPParser,default,gcc,1,0.750839,1,1
1,sym.test_chunked_at_once,0x804aa20,0,1,0.640205,PicoHTTPParser,default,gcc,1,0.784123,1,1
2,sym.test_chunked_leftdata,0x804a320,0,0,0.481506,PicoHTTPParser,default,gcc,1,0.688904,1,1
3,sym.note,0x804a020,0,0,0.496029,PicoHTTPParser,default,gcc,0,0.297617,0,0
4,sym.phr_decode_chunked,0x8049d00,1,0,0.117576,PicoHTTPParser,default,gcc,0,0.070546,0,0


## Testing Recall for Merged LLM and GNN Classifications

In [66]:
def compute_program_recall(df: pd.DataFrame, program_name: str, prediction_column: str = "prediction") -> Dict[str, Any]:
    """
    Compute recall and additional statistics for a specific *program* (not filename).

    Parameters
    ----------
    df : pandas.DataFrame
        The entire prediction dataset loaded from the CSV.
    program_name : str
        Canonical name of the program (matches column 'program').

    Returns
    -------
    Dict[str, Any]
        Dictionary with recall and detailed statistics.
    """

    # Filter rows belonging to this program
    prog_df = df[df["program"] == program_name]

    if prog_df.empty:
        raise ValueError(f"No entries found for program '{program_name}'")

    # ----------------------------------------------------------
    # Convert predicted & true labels to numeric, coercing errors
    # non-numeric labels become NaN and will be excluded automatically
    # ----------------------------------------------------------
    true = pd.to_numeric(prog_df["true_label"], errors="coerce")
    pred = pd.to_numeric(prog_df[prediction_column], errors="coerce")


    # ----------------------------------------------------------
    # Recall computation only on valid numeric entries
    # ----------------------------------------------------------
    valid_mask = (~true.isna()) & (~pred.isna())
    true_valid = true[valid_mask]
    pred_valid = pred[valid_mask]

    tp = ((true_valid == 1) & (pred_valid == 1)).sum()
    actual_pos = (true_valid == 1).sum()

    if actual_pos == 0:
        recall = 1.0
    else:
        recall = tp / actual_pos

    # ----------------------------------------------------------
    # Predicted label statistics
    # ----------------------------------------------------------
    num_pred_0 = (pred_valid == 0).sum()
    num_pred_1 = (pred_valid == 1).sum()
    num_pred_unknown = pred.isna().sum()

    # ----------------------------------------------------------
    # Return statistics dictionary
    # ----------------------------------------------------------
    stats = {
        "program_name": program_name,
        "num_entries": len(prog_df),
        "num_valid_predictions": len(pred_valid),

        # Prediction distribution
        "num_pred_0": num_pred_0,
        "num_pred_1": num_pred_1,
        "num_pred_unknown": num_pred_unknown,

        # Recall metrics
        "true_positives": tp,
        "actual_positives": actual_pos,
        "recall": recall,
    }

    return stats

In [67]:
## Testing Recall for Merged LLM and GNN Classifications
# Weighted fusion Merging Strategy

programs = df_merged["program"].unique()

recalls = [] # Store recall statistics for each program

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df=df_merged, program_name=prog, prediction_column="final_weighted_fusion_prediction")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    recalls.append(stats['recall'])

print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")

-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 760
  Predicted label distribution: 0s=327, 1s=433, unknown=0
  True Positives: 275
  Actual Positives: 334
  Recall: 0.8234
-------------------------------------------------------------------------------------------------------------
Program: Benoitc_HTTP
  Number of entries: 421
  Number of valid predictions: 421
  Predicted label distribution: 0s=277, 1s=144, unknown=0
  True Positives: 63
  Actual Positives: 95
  Recall: 0.6632
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 653
  Predicted label distribution: 0s=335, 1s=318, unknown=0
  True Positives: 165
  Actual Positives: 184
  Recall: 0.8967
--------------------------------------------------------

In [68]:
## Testing Recall for Merged LLM and GNN Classifications
# Hybrid Cascade Merging Strategy

programs = df_merged["program"].unique()

recalls = [] # Store recall statistics for each program

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df=df_merged, program_name=prog, prediction_column="hybrid_cascade")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    recalls.append(stats['recall'])

print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")

-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 760
  Predicted label distribution: 0s=284, 1s=476, unknown=0
  True Positives: 298
  Actual Positives: 334
  Recall: 0.8922
-------------------------------------------------------------------------------------------------------------
Program: Benoitc_HTTP
  Number of entries: 421
  Number of valid predictions: 421
  Predicted label distribution: 0s=253, 1s=168, unknown=0
  True Positives: 95
  Actual Positives: 95
  Recall: 1.0000
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 653
  Predicted label distribution: 0s=307, 1s=346, unknown=0
  True Positives: 178
  Actual Positives: 184
  Recall: 0.9674
--------------------------------------------------------

In [69]:
## Security Check for GNN predictions, should have the same results reported in the Table for the best GNN model

programs = df_merged["program"].unique()

recalls = [] # Store recall statistics for each program

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df=df_merged, program_name=prog, prediction_column="gnn_predicted_label")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    recalls.append(stats['recall'])

print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")

-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 760
  Predicted label distribution: 0s=279, 1s=481, unknown=0
  True Positives: 289
  Actual Positives: 334
  Recall: 0.8653
-------------------------------------------------------------------------------------------------------------
Program: Benoitc_HTTP
  Number of entries: 421
  Number of valid predictions: 421
  Predicted label distribution: 0s=284, 1s=137, unknown=0
  True Positives: 95
  Actual Positives: 95
  Recall: 1.0000
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 653
  Predicted label distribution: 0s=282, 1s=371, unknown=0
  True Positives: 166
  Actual Positives: 184
  Recall: 0.9022
--------------------------------------------------------

In [70]:
## Security Check for LLM predictions, should have the same results reported in the Table for the best LLM model

programs = df_merged["program"].unique()

recalls = [] # Store recall statistics for each program

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df=df_merged, program_name=prog, prediction_column="llm_predicted_label")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    recalls.append(stats['recall'])

print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")

-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 699
  Predicted label distribution: 0s=286, 1s=413, unknown=61
  True Positives: 264
  Actual Positives: 323
  Recall: 0.8173
-------------------------------------------------------------------------------------------------------------
Program: Benoitc_HTTP
  Number of entries: 421
  Number of valid predictions: 308
  Predicted label distribution: 0s=203, 1s=105, unknown=113
  True Positives: 26
  Actual Positives: 58
  Recall: 0.4483
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 604
  Predicted label distribution: 0s=297, 1s=307, unknown=49
  True Positives: 146
  Actual Positives: 165
  Recall: 0.8848
----------------------------------------------------